# EgoEvGesture — Clean, Fixed Heterogeneous Split for Reuse on Kaggle

**Objective:** Generate `train.csv`, `val.csv`, and `test.csv` for VideoMamba-Tiny and baselines, using the **same sample lists for every experiment**. Do not copy the 207 GB event `.npy` files, extract archives, or re-upload the source dataset.

**Proposed protocol** (adapted from the EgoEvGesture heterogeneous split):

| Split | Subjects | Expected event samples before internal deduplication |
|---|---|---:|
| Train | 3, 4, 5, 6, 7 | 2,626 |
| Validation | 8 | 489 |
| Test | 1, 2, 9 | 1,696 |
| Excluded | 10 (100% duplicate of Subject 5, per the previous audit) | 587 |

Before removal, there are **5,398** event files named `actX_Y.npy`. The 87 segmentation-mask `.npy` files **are not classification samples**. Labels use `gesture_id = X % 38` for action IDs `X = 0..75`; `X // 38` identifies the recording condition. **Subject 8 as validation is our choice, not an officially published validation assignment.**

> **Benchmark caveat:** This is a **cleaned benchmark**. A published 62.7% benchmark result is not necessarily directly comparable when training or validation samples differ. Never delete the original dataset.

**Workflow:** 1) Add the raw dataset as Input → 2) Run All → 3) Confirm PASS → 4) Save Version and create a separate Kaggle Dataset from Output. An optional Kaggle CLI publishing cell is provided.

## 1 — Configure and verify Kaggle input paths

Before running, choose **Add Input → Your Work → EgoEvGesture Raw Canonical Archive**. The source root is based on the previously confirmed Kaggle Copy Path.

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import hashlib, json, re, time, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd

SOURCE_REF = 'hoanganh94/egoevgesture-raw'
RAW_ROOT = Path('/kaggle/input/datasets/hoanganh94/egoevgesture-raw')
OUT_DIR = Path('/kaggle/working/egoev_heterogeneous_split_v1')
OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_REF = 'hoanganh94/egoevgesture-heterogeneous-splits-v1'

# TRUE = stop if sample/group counts differ from the audited counts.
STRICT_EXPECTED_COUNTS = True
# TRUE = detect exact duplicates among retained files using fingerprints and SHA-256 as needed.
VERIFY_EXACT_DUPLICATES = True

if not RAW_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset was not added as Input, or its mount changed: {RAW_ROOT}')
print('Raw dataset:', RAW_ROOT)
print('Output CSV:', OUT_DIR)
print('Python:', sys.version.split()[0], 'pandas:', pd.__version__)

## 2 — Index event samples only; exclude masks and annotation files

Inspect `.npy` files **at the top level of each `subjN_left/right` directory**. Ignore nested `mask` and `label` directories. Read only filenames and file sizes; do not load event streams into RAM.

In [ ]:
PATTERN = re.compile(r'^act(?P<action>\d+)_(?P<trial>\d+)\.npy$', re.I)
EXPECTED_COUNTS = {1:675,2:597,3:592,4:302,5:587,6:537,7:608,8:489,9:424,10:587}
rows=[]; bad_names=[]; missing_groups=[]
for subject in range(1, 11):
    for camera in ('left', 'right'):
        group=f'subj{subject}_{camera}'
        directory=RAW_ROOT/group
        if not directory.is_dir():
            missing_groups.append(group)
            continue
        for path in sorted(directory.glob('*.npy')):
            m=PATTERN.fullmatch(path.name)
            if m is None:
                bad_names.append(path.relative_to(RAW_ROOT).as_posix())
                continue
            action=int(m.group('action')); trial=int(m.group('trial'))
            if not 0 <= action <= 75 or trial < 0:
                bad_names.append(path.relative_to(RAW_ROOT).as_posix())
                continue
            rows.append(dict(relative_path=path.relative_to(RAW_ROOT).as_posix(),
                             subject=subject,camera=camera,group=group,
                             action_raw_id=action,trial_id=trial,gesture_id=action % 38,
                             recording_condition=action//38,size_bytes=path.stat().st_size))
if missing_groups: raise RuntimeError('Missing groups: ' + ', '.join(missing_groups))
if bad_names: raise RuntimeError('Invalid event NPY filenames: ' + repr(bad_names[:20]))
index=pd.DataFrame(rows).sort_values('relative_path').reset_index(drop=True)
observed=index.groupby('subject').size().to_dict()
print('Event samples:',len(index))
print('Theo subject:',observed)
print('38 classes:',index.gesture_id.nunique(),'| 2 recording conditions:', index.recording_condition.nunique())
assert index.relative_path.is_unique, 'Duplicate paths'
assert set(index.gesture_id) == set(range(38)), 'One or more of the 38 classes is missing'
if STRICT_EXPECTED_COUNTS:
    assert observed == EXPECTED_COUNTS, f'Counts do not match the audit: {observed}'
    assert len(index)==5398

## 3 — Verify Subject 5/10 duplicate structure and assign splits

The earlier dataset audit verified SHA-256 matches for `subj5_left ↔ subj10_left` and `subj5_right ↔ subj10_right`. This cell compares **filenames and file sizes only**; it does **not** recompute SHA-256 for all 587 pairs. Preserve the original hash audit as supporting evidence.

In [ ]:
s5=index[index.subject==5].copy()
s10=index[index.subject==10].copy()
s5['pair_key']=s5.camera+'|'+s5.relative_path.str.split('/').str[-1]
s10['pair_key']=s10.camera+'|'+s10.relative_path.str.split('/').str[-1]
pair=s5[['pair_key','size_bytes']].merge(s10[['pair_key','size_bytes']],on='pair_key',how='outer',suffixes=('_s5','_s10'),indicator=True)
if len(pair)!=587 or not pair['_merge'].eq('both').all() or not pair.size_bytes_s5.eq(pair.size_bytes_s10).all():
    raise RuntimeError('Subject 10 does NOT match the expected structure of Subject 5; audit again before splitting.')
print('Verified names and sizes for all 587 pairs; SHA-256 equality relies on the earlier audit.')

TRAIN_SUBJECTS={3,4,5,6,7}
VAL_SUBJECTS={8}
TEST_SUBJECTS={1,2,9}
KEEP_SUBJECTS=TRAIN_SUBJECTS|VAL_SUBJECTS|TEST_SUBJECTS
index['split']=index.subject.map(lambda s: 'train' if s in TRAIN_SUBJECTS else ('val' if s in VAL_SUBJECTS else ('test' if s in TEST_SUBJECTS else 'excluded')))
index['exclude_reason']=index['split'].map(lambda v: 'duplicate_of_subject5' if v=='excluded' else '')
selected=index[index.split != 'excluded'].copy().reset_index(drop=True)
assert len(selected)==4811
assert not (set(selected.subject) & {10})
print('Samples remaining after excluding Subject 10:',len(selected))
print(selected.groupby('split').size().to_dict())

## 4 — Detect exact duplicates among the remaining 4,811 samples

For a faster duplicate check, first group files by **identical file size**, then compute **three-region fingerprints**. Only compute full-file SHA-256 for matching fingerprints. Files of different sizes cannot be byte-identical. If a duplicate group crosses **train/val/test boundaries**, stop for investigation; do not silently alter the test partition. Within one split, retain the lexicographically first path and mark the other identical copies as excluded. Stop if duplicate files have conflicting labels.

In [ ]:
def sample_fingerprint(path, size, chunk=256*1024):
    h=hashlib.sha256(b'EgoEv-sample-v1' + str(size).encode('ascii'))
    length=min(chunk, size)
    offsets=sorted(set([0, max(0,(size-length)//2), max(0,size-length)]))
    with path.open('rb') as f:
        for off in offsets:
            f.seek(off)
            data=f.read(length)
            if len(data)!=length:
                raise IOError(f'Short read: {path}')
            h.update(off.to_bytes(8,'little'))
            h.update(data)
    return h.hexdigest()

def full_sha256(path, block=8*1024*1024):
    h=hashlib.sha256()
    with path.open('rb') as f:
        while True:
            b=f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()

hash_log=[]
exact_clusters=[]
if not VERIFY_EXACT_DUPLICATES:
    raise RuntimeError('VERIFY_EXACT_DUPLICATES must remain True for a cleaned benchmark.')
size_counts=selected.size_bytes.value_counts()
candidates=selected[selected.size_bytes.map(size_counts).gt(1)].copy()
print('Equal-size duplicate candidates after excluding Subject 10:',len(candidates))

fingerprints=[]
for j, r in enumerate(candidates.itertuples(index=False),1):
    fingerprints.append(dict(relative_path=r.relative_path,size_bytes=r.size_bytes,
                             fingerprint=sample_fingerprint(RAW_ROOT/r.relative_path, int(r.size_bytes))))
    if j%100==0 or j==len(candidates): print('Fingerprint:',j,'/',len(candidates))
fp=pd.DataFrame(fingerprints,columns=['relative_path','size_bytes','fingerprint'])
if not fp.empty:
    fp_counts=fp.groupby(['size_bytes','fingerprint']).relative_path.transform('size')
    sha_candidates=fp[fp_counts.gt(1)].copy()
else:
    sha_candidates=fp.copy()
print('Files requiring full SHA-256:',len(sha_candidates),'file;',round(sha_candidates.size_bytes.sum()/1e9,3),'GB')

verified=[]
for j,r in enumerate(sha_candidates.itertuples(index=False),1):
    verified.append(dict(relative_path=r.relative_path,size_bytes=r.size_bytes,
                         sha256=full_sha256(RAW_ROOT/r.relative_path)))
    if j%25==0 or j==len(sha_candidates): print('SHA-256:',j,'/',len(sha_candidates))
sha=pd.DataFrame(verified,columns=['relative_path','size_bytes','sha256'])
if not sha.empty:
    for (_,digest),group in sha.groupby(['size_bytes','sha256']):
        if len(group)>1:
            paths=sorted(group.relative_path.tolist())
            meta=selected.set_index('relative_path').loc[paths]
            if len(set(meta['gesture_id']))>1:
                raise RuntimeError('BYTE-IDENTICAL FILES WITH CONFLICTING LABELS! Inspect: '+repr(paths))
            if len(set(meta['split']))>1:
                raise RuntimeError('TRAIN/VAL/TEST LEAKAGE: byte-identical files: '+repr(paths))
            exact_clusters.append(dict(paths=paths,sha256=digest,split=meta.split.iloc[0],
                                       gesture_id=int(meta.gesture_id.iloc[0])))

# Remove only within-split duplicates; never modify the raw dataset.
internal_remove={p for cluster in exact_clusters for p in cluster['paths'][1:]}
print('SHA-256 duplicate clusters in retained data:',len(exact_clusters))
print('Additional samples excluded as within-split duplicates:',len(internal_remove))
if internal_remove:
    selected=selected[~selected.relative_path.isin(internal_remove)].copy()
    index.loc[index.relative_path.isin(internal_remove),'split']='excluded'
    index.loc[index.relative_path.isin(internal_remove),'exclude_reason']='duplicate_within_split'
    print('Example excluded files:',sorted(internal_remove)[:10])
else:
    print('No byte-identical duplicates remain among selected event samples.')

## 5 — Validate the split and export CSVs with provenance

Always store only `relative_path` in the CSV files, never absolute Kaggle mount paths. New notebooks can combine these paths with the raw dataset mount. SHA-256 values in this stage are computed for **duplicate candidates only**, not for all 5,398 source event files.

In [ ]:
assert not selected.relative_path.duplicated().any()
assert set(selected.split)=={'train','val','test'}
assert selected.subject.nunique()==9
assert selected.groupby('subject').split.nunique().eq(1).all()
assert not (set(selected[selected.split=='train'].subject)&set(selected[selected.split=='test'].subject))
assert not (set(selected[selected.split=='val'].subject)&set(selected[selected.split=='test'].subject))
assert not (set(selected[selected.split=='train'].subject)&set(selected[selected.split=='val'].subject))
assert selected.groupby('split').gesture_id.nunique().eq(38).all(), 'A split is missing gesture classes; investigate'
assert (index.split=='excluded').sum()+len(selected)==len(index)

pub_cols=['relative_path','subject','camera','group','action_raw_id','trial_id','gesture_id','recording_condition','size_bytes','split']
selected=selected.sort_values(['split','subject','camera','action_raw_id','trial_id','relative_path']).reset_index(drop=True)
for split in ('train','val','test'):
    part=selected.loc[selected.split==split,pub_cols].sort_values(['subject','camera','action_raw_id','trial_id'])
    part.to_csv(OUT_DIR/(split+'.csv'),index=False)

selected[pub_cols].to_csv(OUT_DIR/'all_splits.csv',index=False)
excluded=index[index.split=='excluded'].sort_values('relative_path').copy()
excluded.to_csv(OUT_DIR/'excluded_samples.csv',index=False)
index.sort_values('relative_path').to_csv(OUT_DIR/'original_event_index.csv',index=False)

ct=pd.crosstab(selected.gesture_id, selected.split).reindex(index=range(38),columns=['train','val','test'],fill_value=0)
ct.index.name='gesture_id'
ct['total']=ct.sum(axis=1)
ct.reset_index().to_csv(OUT_DIR/'class_distribution.csv',index=False)
subj=pd.crosstab(selected.subject, selected.split).reindex(columns=['train','val','test'],fill_value=0)
subj.index.name='subject'
subj.reset_index().to_csv(OUT_DIR/'subject_distribution.csv',index=False)

flat_dup=[]
for c in exact_clusters:
    for p in c['paths']:
        flat_dup.append(dict(relative_path=p,sha256=c['sha256'],split=c['split'],gesture_id=c['gesture_id'],
                             removed_as_duplicate=p in internal_remove))
pd.DataFrame(flat_dup,columns=['relative_path','sha256','split','gesture_id','removed_as_duplicate']).to_csv(OUT_DIR/'verified_internal_duplicates.csv',index=False)

csvfiles=['train.csv','val.csv','test.csv','all_splits.csv','excluded_samples.csv','original_event_index.csv','class_distribution.csv','subject_distribution.csv','verified_internal_duplicates.csv']
file_sha={name:full_sha256(OUT_DIR/name) for name in csvfiles}
manifest={
    'schema_version':'egoev-clean-split/1.0',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_ref':SOURCE_REF,
    'source_mount_example':str(RAW_ROOT),
    'source_event_file_count':int(len(index)),
    'expected_source_event_count':5398,
    'expected_event_file_shape':'N x 4 (not re-read in this notebook)',
    'protocol':'heterogeneous_subject_holdout_cleaned',
    'train_subjects':sorted(TRAIN_SUBJECTS),
    'val_subjects':sorted(VAL_SUBJECTS),
    'test_subjects':sorted(TEST_SUBJECTS),
    'excluded_duplicate_subject':10,
    'subject_5_vs_10_verification':'prior full-file SHA-256 audit; this notebook checks matching filenames and sizes',
    'duplicate_verification':'same-size -> sampled fingerprint -> full SHA-256 for matching fingerprints',
    'exact_duplicate_clusters_after_subject10_drop':len(exact_clusters),
    'extra_duplicate_files_excluded':len(internal_remove),
    'sample_counts':{k:int(v) for k,v in selected.groupby('split').size().items()},
    'class_counts':{k:int(v) for k,v in selected.groupby('split').gesture_id.nunique().items()},
    'relative_paths_only':True,
    'csv_sha256':file_sha,
    'limitations':['Not the authors official validation split','Original source publisher checksums unavailable',
                   'Does not test near-duplicates or subject/camera scene correlation'],
}
(OUT_DIR/'split_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT_DIR/'README.md').write_text("""# EgoEvGesture heterogeneous cleaned splits v1

Source: `hoanganh94/egoevgesture-raw` (the raw event dataset is **not included**).

Protocol: train subjects 3,4,5,6,7; val subject 8; test subjects 1,2,9.
Subject 10 excluded because its event files are byte-for-byte duplicates of subject 5 (prior SHA-256 audit).
Files under `mask/` and `label/` are not used for event classification. Label = action_raw_id % 38.

In a new Kaggle notebook Add Input BOTH this small split dataset and the raw EgoEvGesture dataset.
For every row: `np.load(RAW_ROOT / row.relative_path, mmap_mode="r")`.

The split is research-defined and cleaned, not the author's exact official validation protocol.
See `split_manifest.json` for version, counts, CSV SHA-256 and limitations.
""",encoding='utf-8')

# Optional metadata to enable Kaggle CLI create/auto-version; default PRIVATE.
metadata={
    'id':SPLIT_REF,
    'title':'EgoEvGesture Heterogeneous Cleaned Splits v1',
    'subtitle':'Reproducible train/val/test sample indices for EgoEvGesture; no raw event data',
    'description':'Fixed subject-disjoint CSV splits for EgoEvGesture. Train: 3,4,5,6,7; validation: 8; test: 1,2,9. Subject 10 duplicate excluded. Source raw events must be added separately. License: other; index-only metadata, raw data retains its original publisher license. Review before sharing publicly.',
    'licenses':[{'name':'other'}]
}
(OUT_DIR/'dataset-metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('=== VALIDATION PASS ===')
print('Train/Val/Test:',manifest['sample_counts'])
print('38 classes per split:', manifest['class_counts'])
print('Excluded:',len(excluded),'| including Subject 10:',len(index[index.subject==10]))
print('Output directory:',OUT_DIR)
print('Files:',sorted(p.name for p in OUT_DIR.iterdir()))

## 6 — Verify that the CSVs can be reused for training

Read a few examples from `train.csv` and resolve their paths against the raw event dataset. In later notebooks, add **both datasets** as Input: EgoEvGesture raw and `EgoEvGesture Heterogeneous Cleaned Splits v1`. If Kaggle changes mount layout, update the two `Path` values.

In [ ]:
# Check three CSV samples without loading complete .npy files
train_preview=pd.read_csv(OUT_DIR/'train.csv')
for path in train_preview.relative_path.head(3):
    p=RAW_ROOT/path
    print('FOUND',p.exists(), '|',path)
    assert p.is_file()
print('OK: relative_path resolves against the raw dataset mount.')

## 7 — SAVE ON KAGGLE: recommended UI workflow (no API token required)

1. After seeing `VALIDATION PASS`, choose **Save Version → Save & Run All** to save notebook output and let that version finish.
2. Open the notebook version's **Output → Create Dataset** (or `New Dataset`, depending on the UI). Select the files in `/kaggle/working/egoev_heterogeneous_split_v1/`.
3. Name the new dataset `EgoEvGesture Heterogeneous Cleaned Splits v1`; keep it **Private** until ready to publish. Do **not** include raw `.npy` files.
4. In future training notebooks, **Add Input both the raw dataset and the split CSV dataset**. Kaggle supports creating datasets from notebook outputs.

**Note:** `Run All` creates files under `/kaggle/working`; to preserve them as a separate Kaggle Dataset, use **Create Dataset** from Output or the optional CLI cell below.

Documentation: https://www.kaggle.com/docs/datasets and https://www.kaggle.com/docs/notebooks

### 7B — Optional: publish a Kaggle Dataset directly from the notebook via CLI

Use this only if you have an API token and want to automate publishing. `PUBLISH_VIA_API=False` by default to avoid permission errors or unintended dataset updates. When enabled, the CLI creates a **private** dataset or updates an existing dataset version. Store `KAGGLE_KEY` in Kaggle Secrets; **never paste the token into notebook code**. Authentication may use `KAGGLE_USERNAME`, `KAGGLE_KEY`, or an existing authenticated environment.

In [ ]:
PUBLISH_VIA_API = False  # change to True only if you intentionally use the Kaggle API

if PUBLISH_VIA_API:
    import os, subprocess
    # Load the key from Kaggle Secrets if it is not in the environment.
    if not os.environ.get('KAGGLE_KEY'):
        try:
            from kaggle_secrets import UserSecretsClient
            key=UserSecretsClient().get_secret('KAGGLE_KEY')
            if key:
                os.environ['KAGGLE_KEY']=key
        except Exception as e:
            print('Could not retrieve Kaggle Secrets. Check key and access permissions:', type(e).__name__)
    os.environ.setdefault('KAGGLE_USERNAME',SPLIT_REF.split('/')[0])
    # Do not print environment variables or tokens in notebook output.
    cmd=['kaggle','datasets','status',SPLIT_REF]
    try:
        status=subprocess.run(cmd,capture_output=True,text=True,timeout=90)
        if status.returncode==0:
            upload=['kaggle','datasets','version','-p',str(OUT_DIR),'-m','Validated clean heterogeneous split CSVs']
        elif '404' in (status.stdout+status.stderr) or 'not found' in (status.stdout+status.stderr).lower():
            upload=['kaggle','datasets','create','-p',str(OUT_DIR)] # private by default
        else:
            raise RuntimeError('Could not verify whether the dataset exists (403/auth?): '+(status.stdout+status.stderr)[-600:])
        done=subprocess.run(upload,capture_output=True,text=True,timeout=120)
        print('Publish status:',done.returncode)
        print((done.stdout+done.stderr)[-1200:])
        if done.returncode: raise RuntimeError('CSV upload through API failed; alternatively use Create Dataset from Output.')
        print('Kaggle Dataset:', 'https://www.kaggle.com/datasets/'+SPLIT_REF)
    finally:
        # Clean up temporary credentials only if appropriate; avoid exposing secrets.
        pass
else:
    print('Kaggle API was not called. Publish via Output → Create Dataset as described in step 7.')

## 8 — Use the published CSVs after adding them as Input

```python
from pathlib import Path
import pandas as pd
import numpy as np

RAW_ROOT = Path('/kaggle/input/datasets/hoanganh94/egoevgesture-raw')
SPLIT_ROOT = Path('/kaggle/input/datasets/hoanganh94/egoevgesture-heterogeneous-splits-v1')
train = pd.read_csv(SPLIT_ROOT / 'train.csv')
row = train.iloc[0]
events = np.load(RAW_ROOT / row['relative_path'], mmap_mode='r')
label = int(row['gesture_id'])
print(events.shape, label)
```

All compared models (VideoMamba-Tiny, CNN, EventPoint, and others) should use **the same three CSVs** and deduplication policy. **Do not tune models using the test set.**